### Step 1 – Mount Drive and select project folder

This cell connects Google Drive to the Colab runtime and switches into the TinyML keyword-spotting project directory.

After running it, the workspace is set to desired path

so that all relative paths (like `data_local`) resolve inside this folder.

The printed output lets you quickly verify that:
- The drive is mounted correctly.
- The current working directory matches the project path you expect.
- The directory listing contains the required subfolders (for example, `data_local` for audio, and any scripts or exported headers).

In [ ]:
# Environment setup for Colab:
# - Mount Google Drive so the notebook can access project files.
# - Change the working directory to the TinyML project folder.
# - Print current working directory and its contents for sanity checking.

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Use the BIG dataset (17137 files)
PROJECT_PATH = "/content/drive/MyDrive/opop/TinyML_Absolutist_Keywords"
os.chdir(PROJECT_PATH)

print("CWD:", os.getcwd())
print("Contents:", os.listdir())

### Step 2 – Install packages and import core libraries

This cell installs and imports every library that will be used later in the notebook.

Conceptually, the entire pipeline depends on three components:

1. **Audio + feature extraction**
   - `soundfile` for reading waveforms from `.wav` files.
   - `librosa` for signal processing utilities such as resampling.
2. **Machine learning + evaluation**
   - `numpy` for numerical operations and array handling.
   - `scikit-learn` (`classification_report`, `confusion_matrix`) for metrics.
   - `tensorflow` / `keras` for defining, training, and exporting the neural network.
3. **Visualization**
   - `matplotlib` for plots if you want to visualize loss curves or confusion matrices.

which is useful for reproducibility and debugging (e.g., sharing exact version info when discussing results).

In [ ]:
# Install and import all libraries used in this notebook:
# - Audio I/O and feature extraction (soundfile, librosa)
# - Numerical operations (numpy)
# - Evaluation metrics (scikit-learn)
# - Plotting (matplotlib)
# - Deep learning framework (TensorFlow / Keras)

!pip install -q soundfile librosa scikit-learn matplotlib

import numpy as np
from pathlib import Path
import soundfile as sf
import librosa
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import tensorflow as tf

print("TF version:", tf.__version__)

### Step 3 – Scan the dataset and assign labels

This cell walks through the preprocessed dataset directory `data_local` and builds two parallel arrays:

- `all_paths[i]` – the path to a `.wav` file.
- `all_labels[i]` – the integer label corresponding to that file.

We first define an ordered keyword list

"all", "must", "none", "never", "only"

and a simple mapping indices

For every `.wav` found under `data_local`, we read its parent folder name, map that keyword to the correct integer label, and append both the path and the label. If \(N\) is the total number of usable clips, then

\[
\text{len}(\text{all\_paths}) = \text{len}(\text{all\_labels}) = N.
\]

The `counts` dictionary collects the number of examples per keyword, which is printed at the end. This immediately shows any class imbalance that will later be corrected by oversampling.

In [ ]:
# Scan the preprocessed dataset folder and build:
# - A list of all .wav file paths.
# - Integer labels for each file according to its keyword folder.
# - A count of how many examples exist per keyword.

# Root of your data
ROOT = Path("data_local")

KEYWORDS = ["all", "never", "none", "only", "must"]
key_to_idx = {k: i for i, k in enumerate(KEYWORDS)}

all_paths = []
all_labels = []
counts = {k: 0 for k in KEYWORDS}

print("Scanning data_local ...\n")

for wav in ROOT.rglob("*.wav"):
    label = wav.parent.name.strip().lower()
    if label not in KEYWORDS:
        continue
    all_paths.append(wav)
    all_labels.append(key_to_idx[label])
    counts[label] += 1

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int64)

print("Detected classes & counts:")
for k in KEYWORDS:
    print(f"  {k:5s}: {counts[k]}")

print("\nTotal usable wav files:", len(all_paths))
print("\nLabel mapping:", key_to_idx)

### Step 4 – Shuffle and split into train / validation / test sets

**Context:** We need to split the dataset to ensure the model learns generalized patterns (Training), can be tuned without bias (Validation), and is evaluated fairly on unseen data (Testing).

**Mathematical Formulation:**
Let $\mathcal{D}$ be the full dataset containing $N$ total examples, where $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{N}$. We define split ratios $r_{train}, r_{val}, r_{test}$ such that:

$$
r_{train} + r_{val} + r_{test} = 1
$$

The size of each subset is calculated as:
$$
N_{train} = \lfloor r_{train} \cdot N \rfloor, \quad N_{val} = \lfloor r_{val} \cdot N \rfloor, \quad N_{test} = N - (N_{train} + N_{val})
$$

We apply a random permutation $\pi$ to the indices $\{1, \dots, N\}$ to create randomized subsets:
$$
\mathcal{D}_{train} = \{(x_{\pi(i)}, y_{\pi(i)}) \mid 1 \le i \le N_{train}\}
$$
$$
\mathcal{D}_{val} = \{(x_{\pi(i)}, y_{\pi(i)}) \mid N_{train} < i \le N_{train} + N_{val}\}
$$
$$
\mathcal{D}_{test} = \{(x_{\pi(i)}, y_{\pi(i)}) \mid N_{train} + N_{val} < i \le N\}
$$

In [ ]:
# Randomly shuffle all examples and split indices into:
# - Training set (80%)
# - Validation set (10%)
# - Test set (10%)
# Then index into the path and label arrays to create the three splits.

# Shuffle and split
idx = np.arange(len(all_paths))
np.random.shuffle(idx)

train_ratio = 0.8
val_ratio   = 0.1
test_ratio  = 0.1

N       = len(idx)
n_train = int(N * train_ratio)
n_val   = int(N * val_ratio)
n_test  = N - n_train - n_val

idx_train = idx[:n_train]
idx_val   = idx[n_train:n_train+n_val]
idx_test  = idx[n_train+n_val:]

train_paths  = all_paths[idx_train]
train_labels = all_labels[idx_train]

val_paths    = all_paths[idx_val]
val_labels   = all_labels[idx_val]

test_paths   = all_paths[idx_test]
test_labels  = all_labels[idx_test]

print("Split sizes:")
print("  Train:", len(train_paths))
print("  Val:  ", len(val_paths))
print("  Test: ", len(test_paths))

### Step 5 – Audio loading and time–frequency feature extraction

**Context:** Raw audio is high-dimensional and noisy. We convert it into a compact time-frequency representation (spectrogram-like features) suitable for a small neural network.

**1. Waveform Processing**
We enforce a fixed sample rate $f_s = 16\text{ kHz}$ and duration $t_{clip} = 1.0\text{ s}$. The raw input vector $\mathbf{x}_{raw}$ is padded or trimmed to a fixed length $L$:
$$
L = f_s \times t_{clip} = 16,000 \text{ samples}
$$

**2. Log-Energy Feature Extraction**
Instead of a full FFT, this notebook calculates the "Log Energy" over sliding windows to reduce computational load on the microcontroller.
The waveform is divided into $T$ frames. For each time step $t \in \{1, \dots, T\}$, we extract a window $w_t$ of the signal. The feature value $f_t$ is the log-transformed mean absolute amplitude (energy approximation):

$$
E_t = \frac{1}{|w_t|} \sum_{k \in w_t} |x_k|
$$
$$
X_{t} = \log(1 + E_t)
$$

The resulting feature matrix for one example is replicated across channels to match the expected input shape:
$$
\mathbf{X} \in \mathbb{R}^{T \times F}
$$
where $T=101$ (time steps) and $F=13$ (feature dimensions).

In [ ]:
# Define audio parameters and helper functions that:
# - Load each waveform, convert to mono, and resample to 16 kHz.
# - Pad or trim clips so every example has exactly CLIP_SAMPLES samples.
# - Convert waveforms into a TIME_STEPS x N_FEATS feature matrix.
# - Build feature arrays (X_train, X_val, X_test) for all dataset splits.

from tqdm.auto import tqdm

# ---- Config (must match Arduino later) ----
SAMPLE_RATE   = 16000
CLIP_DURATION = 1.0
CLIP_SAMPLES  = int(SAMPLE_RATE * CLIP_DURATION)

TIME_STEPS = 101   # kFeatureTimeSteps
N_FEATS    = 13    # kFeatureDim

def load_and_pad(path, target_len=CLIP_SAMPLES, sr=SAMPLE_RATE):
    """Load wav, resample to sr, then pad/trim to exactly target_len samples."""
    x, in_sr = sf.read(str(path), dtype="float32")
    if x.ndim > 1:
        x = np.mean(x, axis=1)

    if in_sr != sr:
        x = librosa.resample(x, orig_sr=in_sr, target_sr=sr)

    # pad or trim
    if len(x) < target_len:
        pad = target_len - len(x)
        x = np.pad(x, (0, pad), mode="constant")
    else:
        x = x[:target_len]
    return x

def extract_log_energy_features(x):
    """
    x: 1D waveform of length CLIP_SAMPLES.
    Return features of shape (TIME_STEPS, N_FEATS) using log-energy per frame
    replicated across N_FEATS dims.

    This mirrors the on-device ComputeFeaturesFromGlobalAudio():
      - global RMS normalization to a target RMS
      - per-frame mean(|s|) and log1p
    """
    # ---- Global RMS normalization (match Nano code) ----
    # Assume x is float32 audio (roughly in [-1, 1]).
    rms = np.sqrt(np.mean(np.square(x)) + 1e-12)
    target_rms = 0.10
    if rms > 1e-6:
        scale = target_rms / rms
    else:
        scale = 1.0
    # Prevent crazy amplification
    scale = min(scale, 12.0)

    frame_len = CLIP_SAMPLES // TIME_STEPS
    feats = np.zeros((TIME_STEPS, N_FEATS), dtype=np.float32)

    for t in range(TIME_STEPS):
        start = t * frame_len
        end   = start + frame_len
        if end > CLIP_SAMPLES:
            end = CLIP_SAMPLES
        frame = x[start:end] * scale
        if frame.size == 0:
            energy = 0.0
        else:
            energy = np.mean(np.abs(frame))
        val = np.log1p(energy).astype(np.float32)
        feats[t, :] = val  # replicate across N_FEATS dims

    return feats


def build_feature_array(paths):
    X = np.zeros((len(paths), TIME_STEPS, N_FEATS), dtype=np.float32)
    for i, p in enumerate(tqdm(paths, desc="Extracting features")):
        x = load_and_pad(p)
        feats = extract_log_energy_features(x)
        X[i] = feats
    return X

print("Building training features...")
X_train = build_feature_array(train_paths)
y_train = train_labels.copy()

print("Building validation features...")
X_val = build_feature_array(val_paths)
y_val = val_labels.copy()

print("Building test features...")
X_test = build_feature_array(test_paths)
y_test = test_labels.copy()

print("Shapes:")
print("  X_train:", X_train.shape, " y_train:", y_train.shape)
print("  X_val:  ", X_val.shape,   " y_val:", y_val.shape)
print("  X_test: ", X_test.shape,  " y_test:", y_test.shape)

### Step 6 – Balance the training set by oversampling

**Context:** The dataset may have unequal numbers of samples per keyword. Class imbalance causes the model to become biased toward the majority class. We use **Random Oversampling** to equate the class counts.

**Mathematical Formulation:**
Let $C$ be the set of unique classes. Let $N_c$ be the count of samples for class $c \in C$. We define the target size as the size of the majority class:

$$
N_{max} = \max_{c \in C} (N_c)
$$

For every class $c$, we form a new set $\mathcal{D}'_c$ by sampling $N_{max}$ items from the original set $\mathcal{D}_c$ with replacement:
$$
|\mathcal{D}'_c| = N_{max} \quad \forall c \in C
$$

The final balanced training set is the union of these balanced subsets:
$$
\mathcal{D}_{balanced} = \bigcup_{c \in C} \mathcal{D}'_c
$$

In [ ]:
# Create a class-balanced training set by oversampling:
# - Count how many training examples each class has.
# - Determine the maximum class count.
# - Repeat examples from smaller classes until every class has the same number of samples.
# - Concatenate and shuffle to obtain X_train_bal and y_train_bal.

import numpy as np
from collections import Counter

print("Original class counts in TRAIN:")
cnt = Counter(y_train)
for cls, count in cnt.items():
    print(f"  class {cls}: {count}")

X_train_bal = []
y_train_bal = []

max_per_class = max(cnt.values())
print("\nTarget per-class size:", max_per_class)

for cls in sorted(cnt.keys()):
    idxs = np.where(y_train == cls)[0]
    samples = X_train[idxs]
    labels  = y_train[idxs]

    reps = int(np.ceil(max_per_class / len(samples)))
    samples_rep = np.tile(samples, (reps,1,1))[:max_per_class]
    labels_rep  = np.tile(labels,  reps)[:max_per_class]

    X_train_bal.append(samples_rep)
    y_train_bal.append(labels_rep)

X_train_bal = np.concatenate(X_train_bal, axis=0)
y_train_bal = np.concatenate(y_train_bal, axis=0)

print("\nBalanced TRAIN shape:", X_train_bal.shape, y_train_bal.shape)

# Shuffle
perm = np.random.permutation(len(y_train_bal))
X_train_bal = X_train_bal[perm]
y_train_bal = y_train_bal[perm]

### Step 7 – Compute feature-wise mean and standard deviation

**Context:** Neural networks converge faster and more stably when inputs are centered near zero with unit variance. We compute statistics on the **training set only** and apply them to all sets.

**Computing Statistics:**
We flatten the balanced training tensor into a list of feature vectors. For each feature dimension $j \in \{1, \dots, F\}$, we calculate the mean $\mu_j$ and standard deviation $\sigma_j$:

$$
\mu_j = \frac{1}{M} \sum_{k=1}^{M} x_{k,j}
$$
$$
\sigma_j = \sqrt{\frac{1}{M} \sum_{k=1}^{M} (x_{k,j} - \mu_j)^2 + \epsilon}
$$
*Note: $\epsilon$ is a small constant (1e-8) to prevent division by zero.*

In [ ]:
# Compute feature-wise mean and standard deviation using only the balanced training set:
# - Flatten time dimension so each row is a feature vector.
# - Calculate mean and standard deviation for each of the N_FEATS features.
# - These values will be used to normalize all splits consistently.

# Flatten only TRAINING (balanced)
flat_train = X_train_bal.reshape(-1, X_train_bal.shape[-1])  # (N*101, 13)

feat_mean = flat_train.mean(axis=0)
feat_std  = flat_train.std(axis=0) + 1e-8  # avoid division by zero

print("feat_mean:", feat_mean)
print("feat_std :", feat_std)

### Step 8 – Apply z-score normalization

**Context:** With the feature-wise mean $\boldsymbol{\mu}$ and standard deviation $\boldsymbol{\sigma}$ computed from the balanced training set, we normalize every example in all splits (Train, Validation, and Test).

**Normalization Transform:**
For every input sample $\mathbf{X}$ in the Train, Validation, and Test sets, we apply the transformation element-wise for each time step $t$ and feature $j$:

$$
Z_{i,t,j} = \frac{X_{i,t,j} - \mu_j}{\sigma_j}
$$

In [ ]:
# Normalize training, validation, and test features with z-score normalization:
# - Subtract the training-set mean from every feature.
# - Divide by the training-set standard deviation.
# - Apply exactly the same transform to validation and test data.

def apply_norm(X, mean, std):
    return (X - mean.reshape(1,1,-1)) / std.reshape(1,1,-1)

X_train_n = apply_norm(X_train_bal, feat_mean, feat_std)
X_val_n   = apply_norm(X_val,      feat_mean, feat_std)
X_test_n  = apply_norm(X_test,     feat_mean, feat_std)

print("Normalized shapes:")
print("  X_train_n:", X_train_n.shape)
print("  X_val_n:  ", X_val_n.shape)
print("  X_test_n: ", X_test_n.shape)

### Step 9 – Define the neural network architecture

**Context:** We use a Multi-Layer Perceptron (MLP). This is a feed-forward network consisting of fully connected (Dense) layers.

**Mathematical Formulation:**
The input tensor $\mathbf{X} \in \mathbb{R}^{101 \times 13}$ is first flattened into a vector $\mathbf{v}_0 \in \mathbb{R}^{1313}$. The network processes this vector through layers $l=1, 2, 3$:

1.  **Hidden Layer 1:** (Dense + ReLU)
    $$\mathbf{h}_1 = \text{ReLU}(\mathbf{W}_1 \mathbf{v}_0 + \mathbf{b}_1)$$
2.  **Hidden Layer 2:** (Dense + ReLU)
    $$\mathbf{h}_2 = \text{ReLU}(\mathbf{W}_2 \mathbf{h}_1 + \mathbf{b}_2)$$
3.  **Output Layer:** (Dense / Logits)
    $$\mathbf{z} = \mathbf{W}_3 \mathbf{h}_2 + \mathbf{b}_3$$

Where:
* $\mathbf{W}_l, \mathbf{b}_l$ are the learnable weights and biases.
* $\text{ReLU}(x) = \max(0, x)$ introduces non-linearity.
* $\mathbf{z} \in \mathbb{R}^{C}$ represents the raw **logits** for the $C=5$ classes.

In [ ]:
# Define a small fully connected neural network in Keras:
# - Flatten input feature grid.
# - Two hidden dense layers with ReLU activation.
# - Final dense layer outputs raw logits for each keyword class.
# - Compile with Adam optimizer and sparse categorical cross-entropy loss.

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

TIME_STEPS = 101
N_FEATS    = 13
num_classes = len(np.unique(y_train))  # Should be 5
flat_dim    = TIME_STEPS * N_FEATS

H1 = 64
H2 = 64

model = keras.Sequential([
    keras.Input(shape=(TIME_STEPS, N_FEATS)),
    layers.Flatten(),
    layers.Dense(H1, activation="relu"),
    layers.Dense(H2, activation="relu"),
    layers.Dense(num_classes, activation=None)  # logits
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

model.summary()

### Step 10 – Train the model

**Context:** We train the model by minimizing the discrepancy between the predicted probability distribution and the actual label.

**Mathematical Formulation:**
The model outputs logits $\mathbf{z}$. We convert these to probabilities using the Softmax function:
$$
\hat{y}_c = \text{Softmax}(\mathbf{z})_c = \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}
$$

We minimize the **Sparse Categorical Cross-Entropy Loss**. For a single example with true class index $y$:
$$
\mathcal{L}(\mathbf{z}, y) = -\log(\hat{y}_{y}) = -\log\left( \frac{e^{z_y}}{\sum_{j=1}^{C} e^{z_j}} \right)
$$

The weights $\theta$ are updated using the **Adam Optimizer**, which adapts the learning rate $\eta$ based on the first and second moments of the gradients.

In [ ]:
# Train the neural network:
# - Use normalized, balanced training data as input.
# - Monitor performance on the validation set during training.
# - Run for a fixed number of epochs with a chosen batch size.

EPOCHS = 25
BATCH  = 64

history = model.fit(
    X_train_n, y_train_bal,
    validation_data=(X_val_n, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
)

### Step 11 – Evaluate on the validation set

**Context:** After training, we evaluate how well the model generalizes. We check the accuracy and the confusion matrix on the validation set.

**Confusion Matrix:**
For a set of $N$ validation samples, the confusion matrix $M \in \mathbb{R}^{C \times C}$ is defined such that:

$$
M_{i,j} = \#\{k : y_k = i,\ \hat{y}_k = j\}
$$

This counts how many samples of true class $i$ were predicted as class $j$.

In [ ]:
# Evaluate model performance on the validation set:
# - Generate predictions.
# - Compute precision, recall, and F1-score per class.
# - Show the confusion matrix to inspect which classes are confused.

from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report

print("Train class counts (balanced):")
print(Counter(y_train_bal))

print("\nValidation performance:")
logits_val = model.predict(X_val_n)
y_val_pred = np.argmax(logits_val, axis=1)
print(classification_report(y_val, y_val_pred))

print("Confusion matrix (val):")
print(confusion_matrix(y_val, y_val_pred))

### Step 12 – Evaluate on the held-out test set

**Context:** The test set gives an unbiased estimate of final quality. We look at granular metrics per class.

**Classification Metrics:**
For a specific class $c$, we define $TP_c$ (True Positives), $FP_c$ (False Positives), and $FN_c$ (False Negatives).

$$
\text{Precision}_c = \frac{TP_c}{TP_c + FP_c}, \qquad \text{Recall}_c = \frac{TP_c}{TP_c + FN_c}
$$

The **F1-Score** is the harmonic mean of Precision and Recall:

$$
F1_c = 2 \cdot \frac{\text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}
$$

In [ ]:
# Evaluate model performance on the held-out test set:
# - Compute overall accuracy.
# - Print a detailed classification report.
# - Show the confusion matrix for final model assessment.

logits_test = model.predict(X_test_n)
y_pred = np.argmax(logits_test, axis=1)

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### Step 13 – Extract weights and export a C header

This step takes the trained neural network and converts its learned parameters into a C-compatible header file that can run on an Arduino or any microcontroller.

The model has three Dense layers, each containing a weight matrix and a bias vector:

W1, b1
W2, b2
W3, b3

To reproduce inference on an embedded device, the forward pass must match Keras exactly. The computation performed on the device is:

h1 = ReLU(W1 * x + b1)
h2 = ReLU(W2 * h1 + b2)
z  = W3 * h2 + b3

Because Arduino typically expects matrices in [output][input] format, the code transposes each weight matrix before writing it into the header file.

This cell performs four tasks:
	1.	Extracts W1, b1, W2, b2, W3, b3 from the trained Keras model.
	2.	Transposes each weight matrix to match the microcontroller layout.
	3.	Writes all weights, biases, and model dimensions into the file keyword_linear_model.h.
	4.	Also stores the feature normalization statistics (feat_mean and feat_std), which are required for the C++ preprocessing to match Python exactly.

After this cell runs, the generated header file is ready to be used directly in the Arduino keyword spotting firmware.

In [ ]:
# Extract trained weights from the Keras model and write them to a C header file:
# - Fetch weight matrices and biases from each dense layer.
# - Transpose matrices to match embedded layout.
# - Serialize everything into flat float arrays inside keyword_linear_model.h.

W1, b1 = model.layers[1].get_weights()
W2, b2 = model.layers[2].get_weights()
W3, b3 = model.layers[3].get_weights()

# Transpose to match Arduino layout [out][in]
W1_T = W1.T
W2_T = W2.T
W3_T = W3.T

hidden1_dim = W1_T.shape[0]
hidden2_dim = W2_T.shape[0]

def write_header(path):
    with open(path, "w") as f:
        f.write("// Auto-generated header\n\n")
        f.write("#ifndef KEYWORD_LINEAR_MODEL_H_\n")
        f.write("#define KEYWORD_LINEAR_MODEL_H_\n\n")

        f.write(f"static const int kTimeSteps   = {TIME_STEPS};\n")
        f.write(f"static const int kNumFeatures = {N_FEATS};\n")
        f.write(f"static const int kFlatDim     = {flat_dim};\n")
        f.write(f"static const int kHidden1Dim  = {hidden1_dim};\n")
        f.write(f"static const int kHidden2Dim  = {hidden2_dim};\n")
        f.write(f"static const int kNumClasses  = {num_classes};\n\n")

        # Mean & Std
        f.write("static const float kFeatMean[13] = {")
        f.write(", ".join(f"{v:.8e}f" for v in feat_mean))
        f.write("};\n\n")

        f.write("static const float kFeatStd[13] = {")
        f.write(", ".join(f"{v:.8e}f" for v in feat_std))
        f.write("};\n\n")

        # Write matrices
        def write_matrix(name, M):
            rows, cols = M.shape
            f.write(f"static const float {name}[{rows}][{cols}] = {{\n")
            for r in range(rows):
                f.write("  {")
                f.write(", ".join(f"{v:.8e}f" for v in M[r]))
                f.write("}")
                if r < rows - 1: f.write(",\n")
                else: f.write("\n")
            f.write("};\n\n")

        def write_vector(name, v):
            f.write(f"static const float {name}[{len(v)}] = {{")
            f.write(", ".join(f"{x:.8e}f" for x in v))
            f.write("};\n\n")

        write_matrix("kW1", W1_T)
        write_vector("kB1", b1)
        write_matrix("kW2", W2_T)
        write_vector("kB2", b2)
        write_matrix("kW3", W3_T)
        write_vector("kB3", b3)

        f.write("#endif\n")

write_header("keyword_linear_model.h")
print("Exported: keyword_linear_model.h")

### Step 14 – Download the exported model header

The final cell copies the exported C header file from the Colab environment to your local machine.

The file "keyword_linear_model.h"

contains all network weights, biases, and normalization parameters needed for inference on the microcontroller. Once downloaded, you can include it directly in your Arduino sketch:

```c
#include "keyword_linear_model.h"
```

As long as the on-device preprocessing (sampling rate, framing, log-energy feature extraction, and z-score normalization) matches the notebook implementation, the microcontroller will reproduce the same logits and predictions as the Keras model.

In [ ]:
# Trigger download of the exported C header file so it can be used in an Arduino / embedded TinyML project.

from google.colab import files

files.download("keyword_linear_model.h")
#files.download("confusion_matrix_small.png")
# If you want the Keras model too:
#small_model.save("keyword_small.h5")
#files.download("keyword_small.h5")